# 8.2 File Handling — CSV

**Prerequisites:** 8.1 File Handling — Text  
**Target:** Python 3.12+ (notes flag 3.13/3.14 differences)

### What you'll learn
- What CSV is, and why it is harder than it looks
- 🔴 **Why `newline=""` is required**, and what happens without it
- 🔴 **Why `line.split(",")` is wrong** — quoting and embedded delimiters
- `csv.writer` / `csv.reader` for positional rows
- `csv.DictWriter` / `csv.DictReader` for named columns
- Type conversion — everything comes back as a string
- `csv.Sniffer`, dialects, and streaming large files
- When to stop and use `pandas`

---

## CSV(Comma Seperated Values)
- Tabular form of data can be stored in csv format. 
- CSV format has default delimiter as comma. Other delimiter that we can use is | or \t etc.

### Writing csv file using write():

---

### 🔴 Two things the original notebook used but never explained

#### 1. `newline=""` is not optional

Every `csv` example below passes `newline=""` to `open()`. Here is why.

The `csv` writer emits `\r\n` itself. If you open the file in normal text mode, Python's
newline translation *also* converts `\n` to `\r\n` on Windows — so you get `\r\r\n` and a
**blank line between every record**.

```python
open(path, "w", newline="", encoding="utf-8")     # ✅ always, for CSV
```

The `csv` documentation requires it on **all** platforms, for reading and writing. It is not
a Windows-only workaround.

#### 2. Why you cannot split on commas

The obvious approach — `line.split(",")` — is wrong, because CSV fields may legally contain:

| Case | Example field | Stored as |
|---|---|---|
| A comma | `Widget, large` | `"Widget, large"` |
| A quote | `Bolt 1/2"` | the quote is doubled, and the field is wrapped |
| A newline | two lines of text | `"line one⏎line two"` |

The `csv` module handles the quoting and escaping rules; a `split(",")` never will. This is
the single most common CSV bug, and it only shows up once your data contains real-world text.

In [ ]:
import csv, io
from pathlib import Path

# ---- 🔴 What newline="" actually prevents ----
rows = [["name", "city"], ["alice", "Pune"], ["bob", "Delhi"]]

wrong = Path("File2Save/newline_wrong.csv")
right = Path("File2Save/newline_right.csv")

with open(wrong, "w", encoding="utf-8") as f:          # newline= NOT passed
    csv.writer(f).writerows(rows)

with open(right, "w", newline="", encoding="utf-8") as f:
    csv.writer(f).writerows(rows)

print("without newline='' :", wrong.read_bytes())
print("with    newline='' :", right.read_bytes())
print()
print("without newline='' :", len(wrong.read_text(encoding='utf-8').splitlines()), "lines")
print("with    newline='' :", len(right.read_text(encoding='utf-8').splitlines()), "lines")

wrong.unlink(); right.unlink()

print("""
On Windows the writer emits \\r\\n, then the text layer translates the \\n
again - producing \\r\\r\\n and a blank line between every record.
newline="" turns the text layer's translation off and lets csv do it.
""")


# ---- 🔴 Why you cannot just split on commas ----
tricky = [
    ["id", "description", "price"],
    [1, "Widget, large", 9.99],                   # a comma INSIDE a field
    [2, 'Bolt 1/2"', 0.35],                       # a quote inside a field
    [3, "Multi-line\ndescription", 4.50],         # a NEWLINE inside a field
]

buffer = io.StringIO()
csv.writer(buffer).writerows(tricky)
text = buffer.getvalue()

print("the raw file:")
print(text)

print("naive split(',') on the second data line:")
print("  ", text.splitlines()[1].split(","), " <- the description was torn in two")

print("\ncsv.reader on the same content:")
for row in csv.reader(io.StringIO(text)):
    print("  ", row)
print("\n  ^ commas, quotes AND embedded newlines all survive intact")

In [ ]:
with open('File2Save/tab1.csv','a') as file:
    file.write('Name,Corona Test')
    file.write('\n')
    file.write('Aditya,neg')
    file.write('\n')
    print('Written')

### Define a function for writing on file using write():

In [ ]:
def write_file(name,test):
    with open('File2Save/tab1.csv','a') as file:
        file.write(f'{name},{test}')
    print('Written')    

write_file('Neetu','neg')

In [ ]:
with open('File2Save/tab1.csv') as file:
    for row in file.readlines():
        print(row)

### csv.writer()

In [ ]:
import csv
with open('File2Save/tab2.csv','w',newline='') as file:
    #to avoid newline after every row, we can give newline argument as empty string '' while opening file.
    writer_obj= csv.writer(file)
    # writer() return object type data which will store in variable writer_obj
    writer_obj.writerow(['Name','Corona Test'])
    writer_obj.writerow(['Manish','pos'])
    # writerow() will write single row at a time.
    writer_obj.writerow(['Preeti','neg'])
    writer_obj.writerows([['Bhuwan','pos'],
                          ['Shiv','pos']])
    # writerows() will write multiple rows at a time.
    print('Written')

### csv.DictWriter():

In [ ]:
import csv
with open('File2Save/tab3.csv','w',newline='') as file:
    writer_obj= csv.DictWriter(file,fieldnames=['Name', 'Corona Test'])
    writer_obj.writeheader()
    writer_obj.writerow({'Name':'Manish', 'Corona Test':'pos'})
    writer_obj.writerow({'Corona Test':'neg', 'Name':'Preeti'})
    writer_obj.writerows([{'Name':'Bhuwan', 'Corona Test':'pos'},
                          {'Name':'Shiv', 'Corona Test':'pos'}])
    print("Written")

### csv.reader()

In [ ]:
import csv
with open('File2Save/tab3.csv') as file:
    reader_obj= csv.reader(file) # reader() return iterator object type data.
    print(reader_obj)
    for i in reader_obj:
        print(i) # Each row as List

#### Read csv file excluding header(if present):

In [ ]:
import csv
with open('File2Save/tab3.csv') as file:
    reader_obj= csv.reader(file)
    next(reader_obj)  # next() takes iterator as argument and will move cursor to the next line from first line
    for i in reader_obj:
        print(i)

#### If we want to read only 'Name' column:

In [ ]:
import csv
with open('File2Save/tab3.csv') as file:
    reader_obj= csv.reader(file)
    next(reader_obj)
    for i in reader_obj:
        print(i[0])

#### DictReader():
- Using DictReader class we can read our file in the form of ordered dictionary.

In [ ]:
import csv
with open('File2Save/tab3.csv') as file:
    reader_obj= csv.DictReader(file,delimiter=',')
    #reader_obj will be an instance of class which also
    #work as iterator
    for i in reader_obj:
        print(i) # Each row as Ordered Dictionary

In [ ]:
import csv
with open('File2Save/tab3.csv') as file:
    reader_obj= csv.DictReader(file)
    for i in reader_obj:
        print(i['Name'])

In [ ]:
import csv, io

# ---- 🔴 csv.reader returns STRINGS. Always. ----
raw = "name,age,score,active,notes\nalice,30,91.5,true,\nbob,,88,false,on leave\n"

rows = list(csv.DictReader(io.StringIO(raw)))
print("straight from the reader:")
for row in rows:
    print("  ", {k: (v, type(v).__name__) for k, v in list(row.items())[:3]})


# ---- Converting explicitly, and handling empties ----
def to_int(value: str, default=None):
    return int(value) if value.strip() else default

def to_float(value: str, default=None):
    return float(value) if value.strip() else default

def to_bool(value: str) -> bool:
    return value.strip().lower() in {"true", "yes", "1", "y"}


print("\nafter conversion:")
for row in rows:
    record = {
        "name": row["name"],
        "age": to_int(row["age"]),
        "score": to_float(row["score"]),
        "active": to_bool(row["active"]),
        "notes": row["notes"] or None,
    }
    print("  ", record)


# ---- DictWriter: restval and extrasaction ----
records = [
    {"name": "alice", "age": 30, "city": "Pune"},
    {"name": "bob", "age": 25},                       # missing "city"
    {"name": "carol", "age": 41, "city": "Delhi", "extra": "ignored"},
]

buffer = io.StringIO()
writer = csv.DictWriter(
    buffer,
    fieldnames=["name", "age", "city"],
    restval="N/A",              # what to write when a key is missing
    extrasaction="ignore",      # what to do with keys not in fieldnames
)
writer.writeheader()
writer.writerows(records)

print("\nDictWriter output:")
print(buffer.getvalue())

# Without extrasaction="ignore", the third record raises:
buffer2 = io.StringIO()
strict = csv.DictWriter(buffer2, fieldnames=["name", "age", "city"])
strict.writeheader()
try:
    strict.writerow(records[2])
except ValueError as exc:
    print("strict DictWriter:", exc)

In [ ]:
import csv, io

# ---- csv.Sniffer: work out the format of an unknown file ----
samples = {
    "comma":     "name,age\nalice,30\nbob,25\n",
    "semicolon": "name;age\nalice;30\nbob;25\n",
    "tab":       "name\tage\nalice\t30\nbob\t25\n",
    "pipe":      "name|age\nalice|30\nbob|25\n",
}

for label, text in samples.items():
    dialect = csv.Sniffer().sniff(text)
    has_header = csv.Sniffer().has_header(text)
    rows = list(csv.reader(io.StringIO(text), dialect))
    print(f"  {label:<10} delimiter={dialect.delimiter!r:<5} header={has_header}  {rows[1]}")

print("\n⚠️ Sniffer guesses. Prefer to be TOLD the format; use it only for genuinely")
print("   unknown input, and validate the result.")


# ---- Streaming a large file without loading it ----
big = io.StringIO("region,amount\n" + "\n".join(
    f"{'north' if i % 2 else 'south'},{i * 10}" for i in range(1, 10_001)
))

reader = csv.DictReader(big)
totals: dict[str, int] = {}
count = 0

for row in reader:                       # one row in memory at a time
    totals[row["region"]] = totals.get(row["region"], 0) + int(row["amount"])
    count += 1

print(f"\nstreamed {count:,} rows without materialising them")
for region, total in sorted(totals.items()):
    print(f"  {region:<7} {total:>12,}")

---

### When to stop using the `csv` module

The `csv` module is the right tool for **reading and writing** CSV files. It is the wrong
tool for **analysing** them.

| Task | Use |
|---|---|
| Read/write a few thousand rows | **`csv`** |
| Stream a huge file, row by row | **`csv`** |
| Strict control over quoting and dialect | **`csv`** |
| Filtering, grouping, joining, aggregating | **`pandas`** |
| Numeric analysis, statistics, plotting | **`pandas`** |
| Millions of rows, repeated queries | A **database** (**10**) |

The tell is when your code starts accumulating dictionaries-of-lists to group things. That
is a `pandas` `groupby` in disguise.

```python
import pandas as pd

df = pd.read_csv("sales.csv")
df.groupby("region")["amount"].sum()
```

`pandas` is a third-party package (`pip install pandas`), which is why it is not covered
here — but knowing *when* to reach for it is part of knowing the `csv` module.

---

## Common Mistakes & Pitfalls

1. 🔴 **Omitting `newline=""`.** On Windows you get a blank line between every record. It is required by the `csv` docs on every platform.
2. 🔴 **Splitting CSV lines with `line.split(",")`.** It breaks the moment a field contains a comma, a quote or a newline. Use the `csv` module.
3. **Forgetting `encoding=`.** Same platform-default trap as **8.1**. Use `utf-8`, or `utf-8-sig` for files Excel produced.
4. **Not skipping the header**, so it ends up in your data as a row of strings.
5. **Assuming values are typed.** `csv.reader` returns **strings**, always. `"42"` is not `42`, and `""` is not `None`.
6. **Using `readlines()` on a large CSV.** Iterate the reader instead — it streams.
7. **`DictWriter` without `writeheader()`**, producing a headerless file that `DictReader` then misreads.
8. **Passing a `dict` with extra keys to `DictWriter`** — `ValueError` unless you set `extrasaction="ignore"`.
9. **Hand-rolling analysis on a 500 MB CSV.** At that point reach for `pandas` or a database.

## Best Practices

- **Always** `open(path, newline="", encoding="utf-8")` for CSV — both arguments, every time.
- Use `csv.reader`/`csv.writer` for positional data, `DictReader`/`DictWriter` when the columns have names.
- Call `writeheader()` immediately after creating a `DictWriter`.
- Convert types explicitly on read, and validate as you go.
- Iterate the reader rather than materialising it — CSVs get big.
- Use `utf-8-sig` when the file came from Excel.
- Use `csv.Sniffer` only for genuinely unknown input; prefer to be told the format.
- Reach for `pandas` once you are doing joins, aggregation or numeric analysis.

## Practice Exercises

Try these before moving on.

1. Write a CSV where one field contains a comma and another contains a newline. Read it back with `csv.reader` and confirm it survives. Then try `line.split(",")`.
2. Write the same data with and without `newline=""` and diff the raw bytes.
3. Read a CSV into a list of dicts with numeric columns converted to `int`/`float`, handling empty cells.
4. Use `DictWriter` to write records where some dicts are missing a key — use `restval`.
5. Use `csv.Sniffer` to detect the delimiter of a semicolon-separated file.
6. Stream a large CSV and compute a column total without loading it all into memory.
7. Read a UTF-8 CSV with a BOM using `utf-8` and then `utf-8-sig`. Compare the first key.
8. Write a function that validates each row and collects errors into an `ExceptionGroup` (**6.2**).